# 04.3 · query transfer on top of kv-transfer — isolating the pending-operation computation in the goal heads

Three prompts (letters = sub-goals, `'` = prefilled):
```
p_dst  : a'xy  -> x                                     (clean destination)
p_kv   : a'bc  -> b      k+v of the task list -> DST     (kv-transfer: DST now "sees" tasks a, b, c -> writes b)
p_query: a'b'd -> d      (order-matched)                  pivot query of the goal heads -> DST pivot
         a'db' -> d      (order-diff)
```
Conditions run on DST: **clean** (`x`), **kv-only** (expect `b`), **kv+q** (both). (q-only is skipped: the query alone into the clean DST is expected to be a no-op -> `x`.)

Outcomes of kv+q (judged over {x, y, b, c, d}):
- `x` / `y` — the kv-transfer failed (check against the kv-only row);
- `b` — kv-transfer worked, the query is a no-op;
- `c` — **H1 (the result we are looking for):** the query carried the **progress / set-difference** ({a, b} done) and it was applied to the kv-patched task list {a, b, c};
- `d` — the query carried the source frontier's **identity** (d is not in the kv-patched task list).

Mechanisms are the existing ones: `headwise_kv_patch` (GQA-corrected k/v span transplant, `align_kv_spans` pad) and the q-transfer prefill+decode query clamp (`greedy(prefill_specs, decode_specs, n_steer)`), stacked in one forward. py→py, qualitative.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "3")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("../..")  # repo root, so `src` is importable
import json, random
from contextlib import ExitStack, nullcontext
import torch, numpy as np
import src.diff.functional as P
from src.diff.functional import *          # helpers into scope
from src.diff.functional import _layers_of
import src.diff.data as D

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct") # loads model + head masks (once)
mt = P.mt
from rich.panel import Panel
from rich.text import Text
from rich.markup import escape

with open(os.path.join(os.path.dirname(env_utils.DEFAULT_RESULTS_DIR),
                       "head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json")) as f:
    GOAL_HEADS_250 = [(l, h) for l, h in json.load(f)]

2026-10-05 11:19:45 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


## Knobs

In [2]:
#------------------------------------------------------------------------------------------------------
KV_HEADS = GOAL_HEADS_250     # heads whose k+v over the task list are transplanted from p_kv
Q_HEADS = GOAL_HEADS_250      # heads whose pivot query is transplanted from p_query
READOUT_HEADS = GOAL_HEADS_250
ALPHA = 2.0                   # OV-write boost of KV_HEADS ∪ Q_HEADS at the pivot on the patched runs (1.0 = off)
ALIGN_KNOB = "pad"            # kv span alignment ("pad" | "right")
N_STEER = 16                  # forward passes the query clamp is re-applied for (prefill = 1)
MAX_NEW = 64
DT = "list[str]"              # q-bank datatype: int | str | list[int] | list[str]
#------------------------------------------------------------------------------------------------------

# ── two-span prompt renderer: full task span (bright_yellow) + the EXACT KV window (orange) ────
# The transplant window is chosen by `align_kv_spans(mt, ..., mode=ALIGN_KNOB)` (src.diff.functional).
def show_spans(tok, title, task_span=None, kv_span=None, piv=-1):
    """Render the prompt exactly as the model sees it. task_span -> bright_yellow (the task text);
    kv_span -> `black on dark_orange` (the exact tokens whose k+v are transplanted); pivot -> salmon.
    Priority pivot > kv > task, so the KV window shows through the task highlight."""
    ids = (prepare_input(prompts=tok, tokenizer=mt) if isinstance(tok, str) else tok)["input_ids"][0].tolist()
    n = len(ids)
    pivpos = n + piv if piv < 0 else piv
    task = set(range(*task_span)) if task_span else set()
    kv = set(range(*kv_span)) if kv_span else set()
    # Mark spans with control-char sentinels and escape the WHOLE joined string once (then swap
    # sentinels for tags). Escaping per-token drops brackets like `[w ...]` / `list[str]` when the
    # tokenizer splits `[` from the following letter across two tokens: the lone `[` isn't escaped,
    # so once joined it reads as a markup tag and rich absorbs it.
    PO, PC, KO, KC, TO, TC = "\x00<", "\x00>", "\x01<", "\x01>", "\x02<", "\x02>"
    parts = []
    for i in range(n):
        s = mt.tokenizer.decode([ids[i]]).replace("\n", "\\n\n")
        if i == pivpos:
            s = PO + s + PC
        elif i in kv:
            s = KO + s + KC
        elif i in task:
            s = TO + s + TC
        parts.append(s)
    txt = escape("".join(parts))
    txt = (txt.replace(PO, "[bold black on salmon1]").replace(PC, "[/bold black on salmon1]")
              .replace(KO, "[bold black on dark_orange]").replace(KC, "[/bold black on dark_orange]")
              .replace(TO, "[bright_yellow]").replace(TC, "[/bright_yellow]"))
    rich_print(Panel(Text.from_markup(txt), title=f"[bold]{escape(title)}[/bold]",
                     title_align="left", border_style="grey42", padding=(1, 1)))

## Instances — six distinct q-bank predicates for a, b, c, d, x, y

Same constraints as the hypothesis evals (`src.diff.hyp_eval`): all pairwise compatible (`D.qpred_compatible`), and the judged outcomes {x, y, b, c, d} are non-trivial frontiers, with {x, b, c, d} from distinct semantic families. Prompts are rendered by the eval pipeline's `D.DiffTriggerInstance._parts` (letters naming, `var_eq` pivot).

In [3]:
ROLES = "abcdxy"
JUDGED = "xybcd"
DISTINCT = "xbcd"   # pairwise-distinct semantic families (the list datatypes have only 4; x vs y = both "kv failed")


def sample_roles(dt, n=8, seed=0):
    bank = {p.id: p for p in D.QTASKS[dt].predicates}
    trivial = D.TRIVIAL_FRONTIERS.get(dt, frozenset())
    rng, out, seen = random.Random(seed), [], set()
    for _ in range(n * 5000):
        if len(out) >= n:
            break
        ids = rng.sample(list(bank), len(ROLES))
        r = dict(zip(ROLES, ids))
        judged = [r[k] for k in JUDGED]
        if tuple(ids) in seen or any(i in trivial for i in judged):
            continue
        if len({D._diff_family(dt, r[k]) for k in DISTINCT}) < len(DISTINCT):
            continue
        if not all(D.qpred_compatible(dt, p, q) for k, p in enumerate(ids) for q in ids[k + 1:]):
            continue
        seen.add(tuple(ids))
        out.append({k: bank[v] for k, v in r.items()})
    return out


def render(dt, order, prefilled):
    # (prompt, task-list text) for spec `order` (Predicates) with the `prefilled` letters' lines written
    shell = D.DiffTriggerInstance(datatype=dt, src_order=tuple(order), dst_order=tuple(order), src_progress=[],
                                  dst_progress=[], diff_target="", dst_native="", src_target="",
                                  var_name=D.QTASKS[dt].var_names[0], presentation="listed", src_lang="python",
                                  dst_lang="python", naming="letters", letter_stub="var_eq")
    q, code = shell._parts("python", tuple(order), [p.id for p in prefilled])
    task_text = "\n".join(f"({i + 1}) {p.description}" for i, p in enumerate(order))
    return get_prompt(q, D._finalize_prefill(code)), task_text


def build(roles, q_order="match"):
    r = roles
    p_dst = render(DT, [r["a"], r["x"], r["y"]], [r["a"]])
    p_kv = render(DT, [r["a"], r["b"], r["c"]], [r["a"]])
    q_spec = [r["a"], r["b"], r["d"]] if q_order == "match" else [r["a"], r["d"], r["b"]]
    p_q = render(DT, q_spec, [r["a"], r["b"]])
    return dict(dst=p_dst, kv=p_kv, query=p_q, q_order=q_order)


ROLE_SETS = sample_roles(DT, n=8, seed=0)
for i, r in enumerate(ROLE_SETS):
    print(f"[{i}] " + "  ".join(f"{k}={r[k].id}" for k in ROLES))

[0] a=count_short  b=count_all_lowercase  c=count_ends_consonant  d=shortest_word_length  x=any_empty  y=count_starts_vowel
[1] a=shortest_word_length  b=all_unique_lengths  c=count_capitalized  d=count_ends_consonant  x=has_palindrome  y=distinct_count
[2] a=count_starts_vowel  b=total_vowels  c=shortest_word_length  d=count_capitalized  x=has_palindrome  y=all_unique_lengths
[3] a=count_long  b=has_palindrome  c=longest_word_length  d=count_capitalized  x=total_vowels  y=count_starts_vowel
[4] a=count_long  b=any_empty  c=total_length  d=count_capitalized  x=total_vowels  y=count_ends_consonant
[5] a=count_short  b=count_starts_vowel  c=total_length  d=count_all_lowercase  x=distinct_count  y=total_vowels
[6] a=total_length  b=count_all_lowercase  c=count_with_digit  d=count_long  x=count_ends_consonant  y=longest_word_length
[7] a=all_unique_lengths  b=count_ends_consonant  c=total_length  d=total_uppercase_letters  x=count_with_digit  y=any_empty


## Render one instance — the three prompts (task list = yellow, transplanted k/v window = orange, pivot = salmon)

In [4]:
IDX, Q_ORDER = 0, "match"      # Q_ORDER: "match" (a'b'd) | "diff" (a'db')

R = ROLE_SETS[IDX]
B = build(R, Q_ORDER)
(dst_prompt, dst_task), (kv_prompt, kv_task), (q_prompt, q_task) = B["dst"], B["kv"], B["query"]
KV0, DST0 = token_span(kv_prompt, kv_task), token_span(dst_prompt, dst_task)
kv_tok, dst_tok, KV_POS, DST_POS, KV_SPAN, DST_SPAN = align_kv_spans(
    mt, kv_prompt, dst_prompt, KV0, DST0, mode=ALIGN_KNOB, src_text=kv_task, dst_text=dst_task)
q_tok = prepare_input(prompts=q_prompt, tokenizer=mt)
show_spans(dst_tok, "p_dst  (a'xy -> x) · k/v PATCHED-TO", task_span=DST_SPAN, kv_span=(DST_POS[0], DST_POS[-1] + 1))
show_spans(kv_tok, "p_kv   (a'bc -> b) · k/v CACHED-FROM", task_span=KV_SPAN, kv_span=(KV_POS[0], KV_POS[-1] + 1))
show_spans(q_tok, f"p_query ({'a’b’d' if Q_ORDER == 'match' else 'a’db’'} -> d) · query CACHED-FROM the pivot")

╭─ p_dst  (a'xy -> x) · k/v PATCHED-TO ───────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ (1) the number of words shorter than three characters\n                                                         │
│ (2) whether any word is empty\n                                                                                 │
│ (3) the number of words that start with a vowel\n                                                               │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(1 for w in words if len(w) < 3)\n                                                                   │
│     b =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ p_kv   (a'bc -> b) · k/v CACHED-FROM ──────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of words shorter than three characters\n                                                         │
│ (2) the number of words written entirely in lowercase letters\n                                                 │
│ (3) the number of words that end with a consonant\n                                                             │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(1 for w in words if len(w) < 3)\n                                                                   │
│     b =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ p_query (a’b’d -> d) · query CACHED-FROM the pivot ────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of words shorter than three characters\n                                                         │
│ (2) the number of words written entirely in lowercase letters\n                                                 │
│ (3) the length of the shortest word\n                                                                           │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(1 for w in words if len(w) < 3)\n                                                                   │
│     b = sum(1 for w in words if w.islower())\n                                                                  │
│     c =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Interventions — clean / kv-only / kv+q on DST

`kv` = `headwise_kv_patch` with p_kv's task-list k+v on `KV_HEADS`; `q` = p_query's pivot query on `Q_HEADS` (prefill + `N_STEER` decode clamps). The query goes into the kv-aligned DST. `ALPHA` boosts `KV_HEADS ∪ Q_HEADS` at the pivot on every patched run. GoalScope read-outs at the pivot under the same patches.

In [5]:
def q_specs(q_prompt, piv_dst):
    # prefill + decode query-clamp specs (p_query pivot query -> DST pivot), as in run_q_transfer_intervention
    piv_q = prepare_input(prompts=q_prompt, tokenizer=mt)["input_ids"].shape[1] - 1
    qc = cache_src_proj(q_prompt, [piv_q], sorted({l for l, _ in Q_HEADS}), "q")
    qh = heads_per_layer(Q_HEADS, "q")
    return ([dict(which="q", cache=qc, heads=qh, positions=[piv_dst])],
            [dict(which="q", cache=qc, heads=qh, positions=[-1])])


def kv_caches():
    kvh = heads_per_layer(KV_HEADS, "k", layers=set(_layers_of(KV_HEADS)))
    return cache_src_proj(kv_tok, KV_POS, sorted(kvh), "k"), cache_src_proj(kv_tok, KV_POS, sorted(kvh), "v")


def patched_ctx(kv=None, pf=None, q_boost=False, boost=True):
    # context manager stacking the kv transplant, the (prefill) query clamp and ONE OV boost over the patched
    # heads (KV_HEADS if kv, Q_HEADS if a query patch is active — pf here, or greedy's own clamp: q_boost=True)
    s = ExitStack()
    if kv is not None:
        s.enter_context(headwise_kv_patch(mt, KV_HEADS, kv[0], kv[1], DST_POS))
    if pf is not None:
        s.enter_context(proj_patch_hooks(mt, pf))
    heads = sorted(set(KV_HEADS if kv is not None else []) | set(Q_HEADS if (pf is not None or q_boost) else []))
    if boost and ALPHA != 1.0 and heads:
        s.enter_context(boost_head_ov(mt, heads, ALPHA, positions=(-1,)))
    return s


def ro_iface(kv=None, pf=None, boost=True):
    def _if(mt, inputs, patches=None, model_kwargs=None):   # get_attention_matrices passes mt= by keyword
        with patched_ctx(kv, pf, boost=boost):
            return mt._model(**dict(inputs), **dict(model_kwargs or {}))
    return _if


def first_line(s):
    return s.strip("\n").split("\n")[0]


def run_conditions(verbose=True):
    piv_clean = prepare_input(prompts=dst_prompt, tokenizer=mt)["input_ids"].shape[1] - 1
    piv_al = dst_tok["input_ids"].shape[1] - 1
    kv = kv_caches()
    pf_a, dc_a = q_specs(q_prompt, piv_al)
    out = {}
    out["clean"] = greedy(prepare_input(prompts=dst_prompt, tokenizer=mt), max_new=MAX_NEW)
    out["src_kv"] = greedy(kv_tok, max_new=MAX_NEW)
    out["src_query"] = greedy(q_tok, max_new=MAX_NEW)
    with patched_ctx(kv=kv):
        out["kv"] = greedy(dst_tok, max_new=MAX_NEW)
    with patched_ctx(kv=kv, q_boost=True):   # query clamp via greedy's hooks (prefill + N_STEER decode steps)
        out["kv+q"] = greedy(dst_tok, prefill_specs=pf_a, decode_specs=dc_a, n_steer=N_STEER, max_new=MAX_NEW)
    ro = {
        "clean": goalscope_readout("", dst_prompt, piv_clean, heads=READOUT_HEADS)[0]["patched_gen"],
        "kv": goalscope_readout("", dst_tok, piv_al, heads=READOUT_HEADS, iface=ro_iface(kv=kv))[0]["patched_gen"],
        "kv+q": goalscope_readout("", dst_tok, piv_al, heads=READOUT_HEADS, iface=ro_iface(kv=kv, pf=pf_a))[0]["patched_gen"],
    }
    if verbose:
        rich_print("  ".join(f"[bold]{k}[/bold]={R[k].id}" for k in ROLES) + f"   · query order={Q_ORDER}")
        rich_print(f"  [dim]p_kv clean    : {escape(first_line(out['src_kv']))}[/dim]")
        rich_print(f"  [dim]p_query clean : {escape(first_line(out['src_query']))}[/dim]")
        for k, col in (("clean", "sea_green3"), ("kv", "orange3"), ("kv+q", "salmon1")):
            rich_print(f"  [{col}]{k:<6}[/{col}] gen : {escape(first_line(out[k]))}")
            rich_print(f"  {'':<6} ro  : [italic]{escape(ro[k])}[/italic]")
    return out, ro


GENS, ROS = run_conditions()

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


pos=132 ' =' => I'm checking if any element in the list is an empty string.

KeyboardInterrupt: 

## Gaze — where does the pivot look inside the (kv-patched) task list?

Value-weighted pivot→task attention averaged over `GAZE_HEADS` (as in the q-transfer gaze, measured under the **plain** patch — no OV boost, which scales the write, not the attention). For **kv-only** and **kv+q** the task-span positions hold p_kv's keys/values, so the gaze is read per **p_kv item** (a, b, c), mapped onto DST through the `align_kv_spans` window. **clean** is shown over DST's own items (a, x, y) for reference. Expect kv-only → `b`; kv+q under H1 → `c`.

In [ ]:
from src.attention import get_attention_matrices

GAZE_HEADS = Q_HEADS


def item_spans(prompt, order, letters, pad_shift=0, pad_from=None):
    # {letter: (lo, hi)} token span of each "(i) description" list item (pad-shifted when the side was padded)
    out = {}
    for i, (L, pr) in enumerate(zip(letters, order)):
        lo, hi = token_span(prompt, f"({i + 1}) {pr.description}")
        sh = pad_shift if pad_from is not None and lo >= pad_from else 0
        out[L] = (lo + sh, hi + sh)
    return out


def mean_gaze(attn, positions, span_positions):
    idx = {p: j for j, p in enumerate(positions)}
    ii = [idx[p] for p in span_positions if p in idx]
    return float(np.mean([attn[j] for j in ii])) if ii else float("nan")


def gaze_kvq(verbose=True):
    piv_al = dst_tok["input_ids"].shape[1] - 1
    piv_clean = prepare_input(prompts=dst_prompt, tokenizer=mt)["input_ids"].shape[1] - 1
    kv = kv_caches()
    pf_a, _ = q_specs(q_prompt, piv_al)
    # p_kv items -> DST positions through the aligned window (KV_POS[j] <-> DST_POS[j])
    kv_pad = (KV_SPAN[1] - KV_SPAN[0]) - (KV0[1] - KV0[0])
    kv_items = item_spans(kv_prompt, [R["a"], R["b"], R["c"]], "abc", pad_shift=kv_pad, pad_from=KV0[0])
    to_dst = {k: d for k, d in zip(KV_POS, DST_POS)}
    kv_items_dst = {L: [to_dst[p] for p in range(lo, hi) if p in to_dst] for L, (lo, hi) in kv_items.items()}
    dst_items = item_spans(dst_prompt, [R["a"], R["x"], R["y"]], "axy")
    task_dst = list(range(DST_SPAN[0], DST_SPAN[1]))
    task_clean = list(range(DST0[0], DST0[1]))

    def attn_over(tok, piv, positions, iface=None):
        kw = {} if iface is None else dict(patch_interface=iface)
        info = get_attention_matrices(tok, mt, value_weighted=True, **kw)
        info = info[0] if isinstance(info, list) else info
        return pivot_attn(info, GAZE_HEADS, positions, piv)

    a_clean = attn_over(prepare_input(prompts=dst_prompt, tokenizer=mt), piv_clean, task_clean)
    a_kv = attn_over(dst_tok, piv_al, task_dst, ro_iface(kv=kv, boost=False))
    a_kvq = attn_over(dst_tok, piv_al, task_dst, ro_iface(kv=kv, pf=pf_a, boost=False))
    res = {
        "clean": {L: mean_gaze(a_clean, task_clean, range(*sp)) for L, sp in dst_items.items()},
        "kv": {L: mean_gaze(a_kv, task_dst, ps) for L, ps in kv_items_dst.items()},
        "kv+q": {L: mean_gaze(a_kvq, task_dst, ps) for L, ps in kv_items_dst.items()},
    }
    if verbose:
        from rich.table import Table
        t = Table(title=f"pivot gaze (mean value-weighted attn, {len(GAZE_HEADS)} heads) — argmax in bold", highlight=False)
        t.add_column("condition"); t.add_column("items"); [t.add_column(c, justify="right") for c in ("1st", "2nd", "3rd")]
        for cond, g in res.items():
            top = max(g, key=g.get)
            t.add_row(cond, "a x y" if cond == "clean" else "a b c",
                      *[f"[bold]{L}={v:.4f}[/bold]" if L == top else f"{L}={v:.4f}" for L, v in g.items()])
        rich_print(t)
    return res


GAZE = gaze_kvq()

## Optional — oracle verdict over {x, y, b, c, d}

Same exclusive multi-option judge as the hypothesis evals (`src.diff.hyp_eval._letters_prompt`, shuffled options, "already implemented: a"). Costs one oracle call per condition.

In [ ]:
from src.diff.hyp_eval import _letters_prompt
from src.diff.eval import CONCEPT_SYSTEM
from src.utils.oracle_llms import ask_claude
import re


def judge(text, roles, seed_key=""):
    letters = list(JUDGED)
    random.Random(seed_key).shuffle(letters)
    opts = [(k, roles[k].description) for k in letters]
    raw = ask_claude(_letters_prompt(text, opts, f"computing several properties of `{D.QTASKS[DT].var_names[0]}`",
                                     roles["a"].description,
                                     "the code the model generated to continue the function after the pivot. "
                                     "Judge ONLY the FIRST computation/statement it performs (ignore anything after)"),
                     system_prompt=CONCEPT_SYSTEM, max_tokens=768, model_name="claude-sonnet-4-6")
    v = re.search(r"<verdict>\s*(\d+|NEITHER)\s*</verdict>", raw, re.I)
    if not v:
        return "parse_error"
    return "neither" if v.group(1).upper() == "NEITHER" else letters[int(v.group(1)) - 1]


for k in ("clean", "kv", "kv+q"):
    print(f"{k:<6} -> {judge(GENS[k], R, seed_key=f'{IDX}|{Q_ORDER}|{k}')}")

## Loop — a few instances × both query orders (generations + read-outs; set `ORACLE=True` to judge)

In [ ]:
ORACLE = False
rows = []
for IDX in range(4):
    for Q_ORDER in ("match", "diff"):
        R = ROLE_SETS[IDX]
        B = build(R, Q_ORDER)
        (dst_prompt, dst_task), (kv_prompt, kv_task), (q_prompt, q_task) = B["dst"], B["kv"], B["query"]
        KV0, DST0 = token_span(kv_prompt, kv_task), token_span(dst_prompt, dst_task)
        kv_tok, dst_tok, KV_POS, DST_POS, KV_SPAN, DST_SPAN = align_kv_spans(
            mt, kv_prompt, dst_prompt, KV0, DST0, mode=ALIGN_KNOB, src_text=kv_task, dst_text=dst_task)
        q_tok = prepare_input(prompts=q_prompt, tokenizer=mt)
        rich_print(f"\n[bold]── [{IDX}] {Q_ORDER} ──[/bold]")
        g, ro = run_conditions()
        gz = gaze_kvq()
        if ORACLE:
            rows.append({k: judge(g[k], R, seed_key=f"{IDX}|{Q_ORDER}|{k}") for k in ("clean", "kv", "kv+q")})
            print("  oracle:", rows[-1])